# Extract.ipynb
This file will preprocess and extract neccessary info from `camma_mvor_2018.json` annotation file 

In [1]:
#importing neccessary packages and libraries
import json
from collections import defaultdict
import numpy as np
from utils.general_utils import convertKeypoints, printKp, validateShape, printFrame, printOp 

In [ ]:
#loading in pose annotations 
try:
    with open('camma_mvor_2018.json') as f:
        data = json.load(f)
except FileNotFoundError:
    print("file not found")

except:
    print("other error")

annot_3D = data["annotations3D"] #3D pose annotations


#ensuring annotations where loaded properly
print(annot_3D[0]['keypoints3D'])

In [ ]:
print(len(annot_3D)) #num keypoints

## Validating keypoint shape

Looking at the first annotation entry, we see notice that first keypoint entry has more than 10 keypoints.


In [ ]:
print(len(annot_3D[0]['keypoints3D']))

Given that each keypoint is of the form x,y,z,?  if there were 10 keypoints there should be the lenght of the entry should be 4x10=40. To verify if this is consistent across the whole dataset we will check keypoint count for every entry

In [ ]:
lengths = [len(a['keypoints3D']) for a in annot_3D]
print(set(lengths))  # if one number, it's consistent

The code above proves a disrepency between, documentation and actual data shape. After further inspection, 4th coordinate seems to be irrelevant so it will be dropped from every frame. As for the the 11 and 12 keypoints respectively, after visualization they seem to represent keypoints for the hands, thus they will be left in.

## Reshaping annotations

48 value keypoint will now become a 12x3 list (12 keypoints , xyz coordinates)

In [ ]:
#reshaping all keypoints
for annot in annot_3D:
    annot['keypoints3D']=convertKeypoints(annot['keypoints3D'])




In [ ]:
printKp(annot_3D[0]['keypoints3D'])

## Frame by frame pose grouping 
Relevant information to extraction from each annotation
- `image_ids`
- `person_id`
- `keypoints3D`

Poses per frame will be stored in dictionary with the following shape: 

```python
    poses_per_frame = {
        frame_1: [ {pose1}, ... , {poseN}],
        ...
        frame_N: [ {pose1}, ... , {poseN}],
    }
```



In [ ]:
poses_per_frame = defaultdict(list) 

for annot in annot_3D:
    frame = annot['image_ids']
    person = annot['person_id']
    keypoints = annot['keypoints3D']


    poses_per_frame[frame].append({
        'person': person,
        'keypoints': keypoints
    })

#inspect data to make sure it was parsed properly 
print(f"There's {len(poses_per_frame)} total frames") 
print(f"Frame 1: {list(poses_per_frame.values())[0]}")

multiPersonFrame = 0
for pose in list(poses_per_frame.values()):
    if len(pose)>1:
        multiPersonFrame +=1

print(f"Theres {multiPersonFrame} frames with more than one person")

In [ ]:
for key in poses_per_frame.keys():
    print(key)

In [ ]:
list(poses_per_frame.values())[0:2]

In [ ]:
#checking frame distribution
personFreq = {}

for frame in list(poses_per_frame.values()):
    numPoses = len(frame)
    if numPoses not in personFreq:
        personFreq[numPoses] = 0
    else: 
        personFreq[numPoses] +=1

print(personFreq)

In [ ]:
#checking how many pose annotations in total
total = 0
for pose in list(poses_per_frame.values()):
    total += len(pose)

print(total)

## Per operation grouping
Based on pose grouping above, frames will now be grouped per operation. Operations will be stored in a dictionary with the following shape:

```python
    operations = {
        day_x: {operation_x: [frames]}

    }
```

In [ ]:
OP_RANGES = { #frame span of operations
    'day1': [
        (0, 56)
    ],
    'day2': [
        (0, 117),
        (118, 232),
        (233, 234),
        (235, 245),
        (246, 257),
        (258, 270),
        (271, 278),
        (279, 293),
        (294, 321),
        (322, 324), 
        (325, 329)
    ],
    'day3': [
        (0, 7),
        (8, 28),
        (29, 58),
        (59, 84),
        (85, 124),
        (125, 138),
        (139, 160),
        (161, 172),
        (173, 181),
        (182, 196),
        (197, 217)
        # ommited 3 imgs at end of day 3  because they didnt constitute a full operation
    ],
    'day4': [
        (0, 75),
        (76, 116),
        (117, 121)
    ]
}

operations = defaultdict(lambda: defaultdict(list)) 
def getIndex(imageId): #extracts image index from id 
    return int(imageId.split('_')[0][-6:]) #ex: 10010000002_10020000002_10030000002 --> 000002 --> 2)

def getDay(imageId): #extracts day from id
    day = int(imageId.split('_')[0][0])

    match day:
        case 1:
            return 'day1'
        
        case 2:
            return 'day2'
        
        case 3:
            return 'day3'
        
        case 4:
            return 'day4'
    

#iterating through frames and mapping them to their respective operations 
for frameId, poses in poses_per_frame.items():
    dayString = getDay(frameId)
    frameIdx = getIndex(frameId)


    for opIdx, (opStart, opEnd) in enumerate(OP_RANGES[dayString], start =1):
        
        if opStart <= frameIdx <= opEnd:
            opString = f"operation_{opIdx}"
            operations[dayString][opString].append({
                'frameIdx': frameIdx,
                'poses': poses 
            })
            break #make sure it only maps to one operation

#sort to put frames in order
for day in operations:
    for op in operations[day]:
        operations[day][op].sort(key=lambda f: f['frameIdx'])



### Inspecting data 

In [ ]:
#Visualizing operation annotations
print(f"Day 1, operation 1: {operations['day1']['operation_1']}")

In [ ]:
#inspecting total number of operations 
total = 0
for day in operations:
    for op in operations[day]:
        for frame in operations[day][op]:
            total += len(frame['poses'])

print(total)

## Exporting cleaned annotations

In [ ]:
## exporting per operation grouping to work with cleaner data

with open('poses_per_operation.json', 'w') as f:
    json.dump(operations, f)

In [ ]:
#importing new json
try:
    with open('poses_per_operation.json') as f:
        cleanedAnnots = json.load(f) #cleaned annotations
except FileNotFoundError:
    print("file not found")

except:
    print("other error")

In [ ]:
## converting keypoint arrays into numpy arrays
for day in cleanedAnnots:
    for operation in cleanedAnnots[day]:
        for frame in cleanedAnnots[day][operation]:
            for pose in frame['poses']:
                pose['keypoints'] = np.array(pose['keypoints'])



In [ ]:
#testing validity of keypoint structure
validateShape(cleanedAnnots)

In [ ]:
##exporting cleaned annotations so it can be worked on in gpd notebook
import pickle 

with open('cleanedAnnots.pkl', 'wb') as f:
    pickle.dump(cleanedAnnots, f)
    